In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys
import os
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "utils").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=True)

from utils.seku_crd_corpus_utils import *
from prediction.predict_ner import *
import random

/Work/conda-envs/fire-mixsub/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Corpus Explorartion and Validation from S3 ###

**Check how many papers in S3**

In [6]:
print(AWS_BUCKET_NAME)
print(AWS_DIRECTORY)
stats = count_s3_files(bucket_name=AWS_BUCKET_NAME, directory=AWS_DIRECTORY)
print(stats)

psm-iiser
SEKU-CRD-v1/index_v1

S3 Directory: s3://psm-iiser/SEKU-CRD-v1/index_v1
JSON Files: 617551
Total Size: 105.34 GB
{'total_files': 617551, 'total_size_gb': 105.34}


**Load Main Papers (Paper that refer/cite other papers)**

In [7]:
paper_index = load_paper_index()
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 0line [00:00, ?line/s]

Loading paper index: 166310line [00:25, 6445.86line/s] 


Loaded 166310 papers from /Work/Experiments/citation-processor-ranker/data/unarxive/processed/paper_index_2024.jsonl (6496.9 MB in memory)
Total 166310 Main papers


**Random Paper Pickup and validation**

In [8]:
rand_idx = random.randint(0,nb_main_papers-1)
rand_paper_id = all_main_paper_ids[rand_idx]
print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
main_data = paper_index[rand_paper_id] # data from paper index file
title = main_data["title"]
nb_all_refs = len(main_data["bib_entry_ids"]) # [A] all unfiltered original references
nb_usable_refs = len(main_data["references"]) # [B] all references that have either an openalex or arxiv id, but does not ensure in-corpus membership
nb_inc_refs = len(main_data["in_corpus_references"]) # [C] all references that have either an openalex or arxiv id, and in-corpus membership
ratio = nb_inc_refs / nb_all_refs # [D] in corpus / original unfiltered
print(f'Title: {title}')
print(f"[Derived] All Refs [{nb_all_refs}],Usable Refs [{nb_usable_refs}], In-Corpus_Refs [{nb_inc_refs}], Ratio [{ratio:.2f}]")

# some values are precomputed in paper index, they should have same value as computed above
nb_all_refs_r = main_data["total_bib_entries"] # basically len(A)
nb_inc_refs_r = main_data["nb_reference_in_corpus"] # basically len(C)
ratio_r = main_data["reference_ratio"] # should be same as in D
print(f"[Precomputed] All Refs [{nb_all_refs_r}], In-Corpus_Refs [{nb_inc_refs_r}], Ratio [{ratio_r:.2f}]")

Random Main paper Index:36847 [Corpus ID: 2209.12696]
Title: The shape of non-graviton operators for $SU(2)$
[Derived] All Refs [19],Usable Refs [17], In-Corpus_Refs [15], Ratio [0.79]
[Precomputed] All Refs [19], In-Corpus_Refs [15], Ratio [0.79]


**Checking Paper Content from S3**

In [9]:
in_corpus_refs_ids = main_data["in_corpus_references"] # list of IDs of [C]
# we need to check if these ids truly exists in our Full Paper Data
# our full paper data is in S3 😊, so we will get it from there. 
results = await get_papers_from_s3(in_corpus_refs_ids)
found_full_data = 0
main_paper_data = get_paper_from_s3(rand_paper_id)
main_title,main_abstract = get_title_and_abstract(main_paper_data)
print(f"Checking refernces for Main paper {rand_paper_id}\n{main_title}")
print("="*5)
for paper_id, data in results.items():
    if data is not None:
        ref_title,ref_abstract = get_title_and_abstract(data)
        print(f"{paper_id} found 😁 \n {ref_title}")
        found_full_data += 1
    else:
        print(f"{paper_id} Not found ☹️")
full_data_ratio = found_full_data / nb_all_refs_r
print(f"{full_data_ratio*100:0.2f}% references have full data")

Fetching papers:   0%|          | 0/15 [00:00<?, ?it/s]

Fetching papers: 100%|██████████| 15/15 [00:01<00:00,  7.80it/s]


Checking refernces for Main paper 2209.12696
The shape of non-graviton operators for $SU(2)$
=====
hep-th/0505112 found 😁 
 Five-Dimensional Gauged Supergravity Black Holes with Independent
  Rotation Parameters
hep-th/0003075 found 😁 
 Invasion of the Giant Gravitons from Anti-de Sitter Space
1305.6314 found 😁 
 1/16 BPS States in N=4 SYM
2005.10843 found 😁 
 The growth of the $\frac{1}{16}$-BPS index in 4d $\mathcal{N}=4$ SYM
1810.11442 found 😁 
 Microscopic origin of the Bekenstein-Hawking entropy of supersymmetric
  AdS$_{\bf 5}$ black holes
0803.4183 found 😁 
 Comments on 1/16 BPS Quantum States and Classical Configurations
2111.10720 found 😁 
 Exact QFT duals of AdS black holes
hep-th/0604023 found 😁 
 A Fermi Surface Model for Large Supersymmetric AdS_5 Black Holes
1810.12067 found 😁 
 Large AdS black holes from QFT
0712.2714 found 😁 
 Supergravitons from one loop perturbative N=4 SYM
2005.11240 found 😁 
 AdS black holes and finite N indices
hep-th/0604135 found 😁 
 Giant Magnon

### Model Prediction ###

In [4]:
LOCAL_MODEL_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/model"
WANDB_PROJECT = "psm-iiser-citation-relevancy"
WANDB_ARTIFACT = "qwen-ner-adapter"

In [5]:
tokenizer, model = load_ft_ner_model(local_dir=LOCAL_MODEL_DIR,wandb_project=WANDB_PROJECT,artifact_name=WANDB_ARTIFACT)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/parthosarothi/.netrc.


Looking up 'qwen-ner-adapter:latest' in W&B project 'psm-iiser-citation-relevancy'...

[Download] Downloading from S3...
  Bucket: psm-iiser
  Prefix: ner-model/qwen-ner/090926
  Downloading: ner-model/qwen-ner/090926adapter_model/README.md -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/README.md
  Downloading: ner-model/qwen-ner/090926adapter_model/adapter_config.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/adapter_config.json
  Downloading: ner-model/qwen-ner/090926adapter_model/adapter_model.safetensors -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/adapter_model.safetensors
✓ Download complete: 3 files
✓ Loaded adapter 'qwen-ner-adapter:latest' from s3://psm-iiser/ner-model/qwen-ner/090926


Loading weights: 100%|██████████| 339/339 [01:44<00:00,  3.25it/s]


In [7]:
sample = 25
paper_index = load_paper_index(max_files=sample) # load some papers
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 24line [00:00, 11699.59line/s]

Loaded 25 papers from /Work/Experiments/citation-processor-ranker/data/unarxive/processed/paper_index_2024.jsonl (0.9 MB in memory)
Total 25 Main papers


In [14]:
rand_idx = random.randint(0,sample-1)
rand_paper_id = all_main_paper_ids[rand_idx]
print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
main_data = paper_index[rand_paper_id] # data from paper index file
title = main_data["title"]
print(f'Title: {title}')

Random Main paper Index:0 [Corpus ID: 1802.04591]
Title: First Order Generative Adversarial Networks


In [15]:
main_paper_data = get_paper_from_s3(rand_paper_id)
main_title,main_abstract = get_title_and_abstract(main_paper_data)
result = predict_ner_ft(main_abstract,tokenizer,model,max_new_tokens=2048)

In [16]:
print(main_abstract)

GANs excel at learning high dimensional distributions, but they can update
generator parameters in directions that do not correspond to the steepest
descent direction of the objective. Prominent examples of problematic update
directions include those used in both Goodfellow's original GAN and the
WGAN-GP. To formally describe an optimal update direction, we introduce a
theoretical framework which allows the derivation of requirements on both the
divergence and corresponding method for determining an update direction, with
these requirements guaranteeing unbiased mini-batch updates in the direction of
steepest descent. We propose a novel divergence which approximates the
Wasserstein distance while regularizing the critic's first order information.
Together with an accompanying update direction, this divergence fulfills the
requirements for unbiased steepest descent updates. We verify our method, the
First Order GAN, with image generation on CelebA, LSUN and CIFAR-10 and set a
new state 

In [17]:
result

[{'text': 'GANs', 'label': 'KEY_CONCEPT', 'start': 0, 'end': 4},
 {'text': 'high dimensional distributions',
  'label': 'KEY_CONCEPT',
  'start': 23,
  'end': 53},
 {'text': 'steepest\ndescent direction',
  'label': 'KEY_CONCEPT',
  'start': 140,
  'end': 166},
 {'text': "Goodfellow's original GAN",
  'label': 'METHOD',
  'start': 264,
  'end': 289},
 {'text': 'WGAN-GP', 'label': 'METHOD', 'start': 298, 'end': 305},
 {'text': 'theoretical framework',
  'label': 'CONTRIBUTION',
  'start': 372,
  'end': 393},
 {'text': 'optimal update direction',
  'label': 'OBJECTIVE',
  'start': 331,
  'end': 355},
 {'text': 'divergence', 'label': 'KEY_CONCEPT', 'start': 450, 'end': 460},
 {'text': 'method', 'label': 'KEY_CONCEPT', 'start': 479, 'end': 485},
 {'text': 'unbiased mini-batch updates',
  'label': 'OBJECTIVE',
  'start': 560,
  'end': 587},
 {'text': 'First Order GAN',
  'label': 'CONTRIBUTION',
  'start': 912,
  'end': 927},
 {'text': 'image generation', 'label': 'OBJECTIVE', 'start': 934,

In [ ]:
abstract = """With the rapid growth of Web-based academic publications, more
and more papers are being published annually, making it increasingly difficult to find relevant prior work. Citation prediction aims
to automatically suggest appropriate references, helping scholars
navigate the expanding scientific literature. Here we present CiteRAG, the first comprehensive retrieval-augmented generation
(RAG)-integrated benchmark for evaluating large language models
on academic citation prediction, featuring a multi-level retrieval
strategy, specialized retrievers, and generators. Our benchmark
makes four core contributions: (1) We establish two instances of the
citation prediction task with different granularity. Task 1 focuses on
coarse-grained list-specific citation prediction, while Task 2 targets
fine-grained position-specific citation prediction. To enhance these
two tasks, we build a dataset containing 7,267 instances for Task
1 and 8,541 instances for Task 2, enabling comprehensive evaluation of both retrieval and generation. (2) We construct a three-level
large-scale corpus with 554k papers spanning many major subfields,using an incremental pipeline. (3) We propose a multi-level hybrid
RAG approach to citation prediction, fine-tuning embedding models
with contrastive learning to capture complex citation relationships,
paired with specialized generation models. (4) We conduct extensive experiments across state-of-the-art language models, including
closed-source APIs, open-source models, and our fine-tuned generators, demonstrating the effectiveness of our framework. Our
open-source toolkit enables reproducible evaluation and focuses on
academic literature, providing the first comprehensive evaluation
framework for citation prediction and serving as a methodological
template for other scientific domains. Our source code and data are
released at https://github.com/LQgdwind/CiteRAG."""

result = predict_ner_ft(abstract,tokenizer,model)